# Cache Speed Benchmark

Pure speed comparison of cached vs streaming feature extraction.
No permutation tests, no reports - just raw extraction timing.

In [1]:
import sys
import time
from pathlib import Path
from datetime import datetime

project_root = Path.cwd() if (Path.cwd() / 'vault').exists() else Path.cwd().parent
sys.path.insert(0, str(project_root))

from utils.enums import Ticker, TimeFrame
from feature_extraction.feature_extractor import extract_features_for_bias_node

print(f"Project root: {project_root}")

Project root: c:\Users\adabla\Trading-Algo


In [2]:
# Configuration
TICKERS = [Ticker.ES, Ticker.NQ, Ticker.YM, Ticker.RTY]
START_DATE = datetime(2000, 1, 1)
END_DATE = datetime(2024, 12, 31)

# All bias nodes from the research notebook
BIAS_NODES = [
    {'name': 'Adaptive_RSI', 'spec': {'module_name': 'adaptive_rsi', 'timeframes': [TimeFrame.D], 'params': {'min_period': [2, 3], 'max_period': [10, 14], 'volatility_period': [20]}}},
    {'name': 'Casey_C', 'spec': {'module_name': 'casey_c', 'timeframes': [TimeFrame.D], 'params': {'change_period': [5, 10], 'ranking_period': [50, 100]}}},
    {'name': 'Cyclical_RSI', 'spec': {'module_name': 'cyclical_rsi', 'timeframes': [TimeFrame.D], 'params': {'short_period': [5], 'long_period': [20], 'rsi_period': [14]}}},
    {'name': 'Detrended_RSI', 'spec': {'module_name': 'detrended_rsi', 'timeframes': [TimeFrame.D], 'params': {'rsi_period': [14], 'detrend_period': [50, 100]}}},
    {'name': 'DeMark_REI', 'spec': {'module_name': 'demark_rei', 'timeframes': [TimeFrame.D], 'params': {'period': [5, 8, 13]}}},
    {'name': 'RSI_Percentile', 'spec': {'module_name': 'rsi_percentile', 'timeframes': [TimeFrame.D], 'params': {'rsi_period': [14], 'percentile_period': [126, 252]}}},
    {'name': 'Stochastic_RSI', 'spec': {'module_name': 'stochastic_rsi', 'timeframes': [TimeFrame.D], 'params': {'rsi_period': [14], 'stoch_period': [14], 'smooth_k': [3]}}},
    {'name': 'TSI', 'spec': {'module_name': 'tsi', 'timeframes': [TimeFrame.D], 'params': {'long_period': [25], 'short_period': [13]}}},
    {'name': 'ZScore_RSI', 'spec': {'module_name': 'zscore_rsi', 'timeframes': [TimeFrame.D], 'params': {'rsi_period': [14], 'zscore_period': [50, 100]}}},
    {'name': 'Percent_B', 'spec': {'module_name': 'percent_b', 'timeframes': [TimeFrame.D], 'params': {'period': [20], 'std_dev': [2.0], 'lower_threshold': [0.0, 0.2]}}},
    {'name': 'Double_7s', 'spec': {'module_name': 'double7s', 'timeframes': [TimeFrame.D], 'params': {'short_period': [7], 'ma_period': [200]}}},
    {'name': 'RSI_Signal', 'spec': {'module_name': 'rsi_signal', 'timeframes': [TimeFrame.D], 'params': {'rsi_period': [14], 'oversold': [30], 'overbought': [70], 'mode': ['threshold']}}},
    {'name': 'SuperTrend_Cross', 'spec': {'module_name': 'supertrend_cross', 'timeframes': [TimeFrame.D], 'params': {'atr_period': [10], 'fast_multiplier': [2.0], 'slow_multiplier': [3.0]}}},
]

print(f"Tickers: {[t.name for t in TICKERS]}")
print(f"Date range: {START_DATE.date()} to {END_DATE.date()}")
print(f"Bias nodes: {len(BIAS_NODES)}")

Tickers: ['ES', 'NQ', 'YM', 'RTY']
Date range: 2000-01-01 to 2024-12-31
Bias nodes: 13


## CACHED Extraction (use_cache=True)

In [3]:
print("="*70)
print("CACHED EXTRACTION")
print("="*70)

cached_times = {}
total_cached_time = 0

for i, node in enumerate(BIAS_NODES, 1):
    name = node['name']
    spec = node['spec']
    
    t0 = time.time()
    try:
        features, targets = extract_features_for_bias_node(
            bias_spec=spec,
            ticker=TICKERS,
            start=START_DATE,
            end=END_DATE,
            use_cache=True
        )
        elapsed = time.time() - t0
        cached_times[name] = elapsed
        total_cached_time += elapsed
        print(f"[{i:2d}/{len(BIAS_NODES)}] {name:20s} {elapsed:6.2f}s  ({features.shape[0]} rows, {features.shape[1]} cols)")
    except Exception as e:
        print(f"[{i:2d}/{len(BIAS_NODES)}] {name:20s} FAILED: {e}")
        cached_times[name] = None

print(f"\nTotal CACHED time: {total_cached_time:.2f}s")

CACHED EXTRACTION
[ 1/13] Adaptive_RSI           1.58s  (20294 rows, 9 cols)
[ 2/13] Casey_C                0.46s  (20294 rows, 9 cols)
[ 3/13] Cyclical_RSI           0.35s  (20294 rows, 6 cols)
[ 4/13] Detrended_RSI          0.42s  (20294 rows, 7 cols)
[ 5/13] DeMark_REI             0.41s  (20294 rows, 8 cols)
[ 6/13] RSI_Percentile         0.38s  (20294 rows, 7 cols)
[ 7/13] Stochastic_RSI         0.34s  (20294 rows, 6 cols)
[ 8/13] TSI                    0.33s  (20294 rows, 6 cols)
[ 9/13] ZScore_RSI             0.38s  (20294 rows, 7 cols)
[10/13] Percent_B              0.37s  (20294 rows, 7 cols)
[11/13] Double_7s              0.33s  (20294 rows, 6 cols)
[12/13] RSI_Signal             0.33s  (20294 rows, 6 cols)
[13/13] SuperTrend_Cross       0.36s  (20294 rows, 6 cols)

Total CACHED time: 6.04s


## STREAMING Extraction (use_cache=False)

**Warning:** This will be slow (~8+ minutes)

In [4]:
print("="*70)
print("STREAMING EXTRACTION (this will take several minutes)")
print("="*70)

streaming_times = {}
total_streaming_time = 0

for i, node in enumerate(BIAS_NODES, 1):
    name = node['name']
    spec = node['spec']
    
    t0 = time.time()
    try:
        features, targets = extract_features_for_bias_node(
            bias_spec=spec,
            ticker=TICKERS,
            start=START_DATE,
            end=END_DATE,
            use_cache=False
        )
        elapsed = time.time() - t0
        streaming_times[name] = elapsed
        total_streaming_time += elapsed
        print(f"[{i:2d}/{len(BIAS_NODES)}] {name:20s} {elapsed:6.2f}s  ({features.shape[0]} rows, {features.shape[1]} cols)")
    except Exception as e:
        print(f"[{i:2d}/{len(BIAS_NODES)}] {name:20s} FAILED: {e}")
        streaming_times[name] = None

print(f"\nTotal STREAMING time: {total_streaming_time:.2f}s")

STREAMING EXTRACTION (this will take several minutes)
[ 1/13] Adaptive_RSI          10.35s  (20294 rows, 9 cols)
[ 2/13] Casey_C                8.93s  (20294 rows, 9 cols)
[ 3/13] Cyclical_RSI           7.29s  (20294 rows, 6 cols)
[ 4/13] Detrended_RSI          7.68s  (20294 rows, 7 cols)
[ 5/13] DeMark_REI             6.89s  (20294 rows, 8 cols)
[ 6/13] RSI_Percentile         8.72s  (20294 rows, 7 cols)
[ 7/13] Stochastic_RSI         8.12s  (20294 rows, 6 cols)
[ 8/13] TSI                    7.29s  (20294 rows, 6 cols)
[ 9/13] ZScore_RSI             7.21s  (20294 rows, 7 cols)
[10/13] Percent_B              9.19s  (20294 rows, 7 cols)
[11/13] Double_7s              6.76s  (20294 rows, 6 cols)
[12/13] RSI_Signal             6.73s  (20294 rows, 6 cols)
[13/13] SuperTrend_Cross       7.24s  (20294 rows, 6 cols)

Total STREAMING time: 102.41s


## Summary Comparison

In [5]:
import pandas as pd

print("="*70)
print("SPEED COMPARISON SUMMARY")
print("="*70)

comparison_data = []
for name in cached_times.keys():
    cached = cached_times.get(name)
    streaming = streaming_times.get(name)
    if cached and streaming:
        speedup = streaming / cached
        comparison_data.append({
            'Node': name,
            'Cached (s)': f'{cached:.2f}',
            'Streaming (s)': f'{streaming:.2f}',
            'Speedup': f'{speedup:.1f}x'
        })

df = pd.DataFrame(comparison_data)
print(df.to_string(index=False))

print(f"\n{'='*70}")
print(f"TOTAL CACHED:    {total_cached_time:7.2f}s")
print(f"TOTAL STREAMING: {total_streaming_time:7.2f}s")
overall_speedup = total_streaming_time / total_cached_time if total_cached_time > 0 else 0
print(f"OVERALL SPEEDUP: {overall_speedup:.1f}x")
print(f"{'='*70}")

SPEED COMPARISON SUMMARY
            Node Cached (s) Streaming (s) Speedup
    Adaptive_RSI       1.58         10.35    6.5x
         Casey_C       0.46          8.93   19.4x
    Cyclical_RSI       0.35          7.29   20.8x
   Detrended_RSI       0.42          7.68   18.4x
      DeMark_REI       0.41          6.89   16.8x
  RSI_Percentile       0.38          8.72   23.2x
  Stochastic_RSI       0.34          8.12   23.7x
             TSI       0.33          7.29   22.0x
      ZScore_RSI       0.38          7.21   19.1x
       Percent_B       0.37          9.19   25.1x
       Double_7s       0.33          6.76   20.3x
      RSI_Signal       0.33          6.73   20.5x
SuperTrend_Cross       0.36          7.24   20.0x

TOTAL CACHED:       6.04s
TOTAL STREAMING:  102.41s
OVERALL SPEEDUP: 17.0x
